# Particle track finding as an integer program

*nuclear · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/ilp_track_finding)

Real hits from a simulated HL-LHC collision (TrackML challenge, CC BY 4.0) are joined into tracks by an integer linear program: choose hit triplets with consistent curvature and polar angle, with each hit used at most once. Our tensor PDLP (restarted primal–dual hybrid gradient, the LP method inside OR-Tools and cuOpt) solves the LP relaxation live on sparse tensors, then rounds and repairs it. It is compared with a hand-written Hungarian (Kuhn–Munkres) baseline and with offline runs of HiGHS, Gurobi and scipy: objective gap, efficiency, fake rate, the official TrackML score and the wall-time scaling $t \propto N^k$.

**Vector form**

$$\max_{x,y\in\{0,1\}}\ \mathbf s^\top\mathbf x - \lambda\,\mathbf 1^\top\mathbf y\quad\text{s.t.}\quad A\begin{pmatrix}\mathbf x\\ \mathbf y\end{pmatrix}\le\mathbf b,\qquad s_t = 1 - \Big(\frac{\Delta\kappa_t}{\sigma_\kappa}\Big)^2 - \Big(\frac{\Delta\theta_t}{\sigma_\theta}\Big)^2,\qquad \mathbf z^{k+1} = \Pi_{[0,1]}\big(\mathbf z^k - \tau(A^\top\mathbf w^k - \mathbf c)\big),\ \ \mathbf w^{k+1} = \big(\mathbf w^k + \sigma(A(2\mathbf z^{k+1} - \mathbf z^k) - \mathbf b)\big)_+$$

**Index (tensor) form**

$$\sum_{e\in\mathrm{out}(h)} y_e \le 1,\quad \sum_{e\in\mathrm{in}(h)} y_e \le 1,\quad \sum_{t:\,e_1(t)=e} x_t \le y_e,\quad \sum_{t:\,e_2(t)=e} x_t \le y_e,\qquad g_v = A_{mv}w_m - c_v,\qquad r_m = A_{mv}(2z^{k+1}_v - z^k_v) - b_m$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $(x_h, y_h, z_h),\ l_h$ | `load_event(); wedge(event, dphi)` | (N, 3), (N,) | real TrackML hits on the 4 pixel-barrel layers, cut to a $\Delta\phi$ wedge: the problem size |
| $e = (a \to b)$ | `segments(xyz, layer, cuts)` | (E, 2) | hit pairs on adjacent layers with $/\Delta\phi/ \le \arcsin(r_b/2R) - \arcsin(r_a/2R)$ and $/z_0/ \le 150$ mm, all pairs at once |
| $\kappa_e$ | `2 * cross / (\|a\| \|b\| \|b - a\|)` | (E,) | curvature of the circle through the beam line and both hits |
| $t = (e_1, e_2),\ s_t$ | `join(seg, n); triplet_score(...)` | (T, 2), (T,) | triplets share the middle hit; a helix has the same $\kappa$ and $\theta$ on both halves, so $s_t$ drops with the mismatch |
| $A_{mv}$ | `torch.sparse_coo_tensor(...)` | (M, T+E) | hit-exclusivity rows (one segment in, one out) and continuity rows; repeated $v$ ⇒ summed in every product |
| $z^{k+1},\ w^{k+1}$ | `pdlp(a, b, c)` | (V,), (M,) | PDHG on the Ruiz-scaled LP with averaging, restarts and a primal weight; two sparse mat-vecs per iteration |
| $x, y \in \{0,1\}$ | `round_and_repair(z, seg, trip, score, n)` | (T,), (E,) | rank segments by LP value, accept conflict-free ones in parallel rounds, trim chain ends that cost more than they score |
| $\min_\pi \sum_i C_{i\pi(i)}$ | `hungarian(cost)` | (n,) | Kuhn–Munkres with potentials: one exact linear assignment per layer pair, chained into tracks |
| $S$ | `trackml_score(label, particle, weight)` | () | official TrackML score: summed weights of hits on tracks that hold > 50% of a particle and are > 50% that particle |

Status: a tool comparison, not an open problem. The ILP is a set-packing problem (NP-hard in general); PDLP is a first-order LP method, and rounding gives a heuristic integer solution, not a certified optimum, so the gap to the exact optimum is always reported. The PDLP dual bound gives a certified upper bound even without an exact solver. Method. Candidate segments join hits on adjacent layers that a track with $p_T \ge p_{T,\min}$ from the beam line could connect. Triplets join two segments through a shared middle hit and score $s_t = 1 - (\Delta\kappa/\sigma_\kappa)^2 - (\Delta\theta/\sigma_\theta)^2$ (only $s_t > 0$ kept). The ILP picks triplets $x_t$ and segments $y_e$: each hit has at most one chosen segment in and one out, so tracks are chains. A chosen segment continues into at most one chosen triplet on each side, and every segment costs $\lambda = 0.1$. Its LP relaxation ($0 \le x, y \le 1$) is solved by our PDLP (Applegate et al. 2021): PDHG with Ruiz scaling, KKT-based restarts to the average and primal-weight updates, written with torch.sparse so it runs on CPU or CUDA. The Hungarian baseline solves one exact linear assignment per layer pair (middle pair first, then inwards and outwards, with triplet costs given the chain so far). It is a bipartite matching solved exactly in polynomial time, but it decides one layer pair at a time, while the ILP weighs whole chains globally. That is why its tracks differ. Data. TrackML Particle Tracking Challenge (CERN / Kaggle 2018), training event 1000 (200 pile-up collisions): Zenodo record 4730167, doi:10.5281/zenodo.4730167, licence CC BY 4.0 (Salzburger, Innocente, Vlimant, Rousseau et al.). It is cut to the pixel barrel (volume 8, 4 layers) with $|\eta| < 1$: 11 519 hits, 2 452 particles and 6% noise, plus the official per-hit weights. Offline benchmark (data/solver_benchmark.csv) on an AMD Ryzen 5 5600G (1 thread): HiGHS through scipy 1.18.1 (milp, linprog), Gurobi 13 through gurobipy with its size-limited licence (at most 2000 variables and 2000 constraints, so only $\Delta\phi \le 30°$), scipy's linear_sum_assignment, and our PDLP on the CPU and on an NVIDIA GTX 1080 Ti. NVIDIA cuOpt 26.10 installed in WSL but could not run: its kernels are built for compute capability 7.0 (Volta) and newer, and the 1080 Ti is 6.1 (cudaErrorNoKernelImageForDevice), so it is skipped and has no numbers. The cited state of the art is in data/trackml_sota.csv. Verdict. On all eight wedges (5° to 360°, 151 to 11 519 hits) the LP relaxation was tight: the HiGHS LP optimum equals the MILP optimum to $10^{-13}$. Our PDLP plus repair reached the same optimum every time (gap below $10^{-9}$), with exactly the triplets HiGHS and Gurobi chose (0 differ). That is empirical, not guaranteed: set packing is NP-hard, and here the four-layer chain structure keeps the LP tight. With continuous scores exact ties are unlikely, so solvers that reach the optimum agree. Mid-run, about a third of the PDLP values are fractional, many near $\tfrac12$ where rival triplets almost tie (for example double hits in overlapping modules); they settle as PDLP converges. The Hungarian chains are 0 to 2.2% below the optimum (5 to 651 triplets differ) and match scipy's assignment objective exactly (a check of our Hungarian). Full barrel subset (360°): efficiency 93.1% and fake rate 13.0% for the ILP solvers vs 90.9% and 16.2% for Hungarian, TrackML score 0.495 vs 0.482. A perfect tracker of the reconstructable particles would score 0.618 here; the cited full-event state of the art is 0.88 to 0.94 (Mikado 0.944 in 0.56 s). Scaling over $N \ge 1000$ hits: HiGHS MILP $N^{1.14}$, HiGHS LP $N^{1.01}$, our PDLP on the CPU $N^{0.92}$, on the GPU $N^{0.36}$ (launch overhead still dominates at 10 000 variables, so the 1080 Ti is not faster yet), our Hungarian $N^{2.14}$ (worst case $O(n^3)$; sparse, gated costs keep augmenting paths short), scipy $N^{1.57}$. Gurobi fits in its size limit only up to 860 hits ($N^{1.04}$ there). At these sizes HiGHS is as fast as PDLP; first-order methods pay off on much larger LPs. On small wedges Hungarian finds slightly more particles (97.4% vs 94.8% at 30°): the ILP objective is a proxy, not the physics metric. Caveats. Our instances are a barrel wedge with 4 layers, so tracks have at most 4 hits and there are no endcaps: easier and different from the full events of the challenge. The TrackML score comparison is indicative, not like for like: about 40% of the subset's score weight sits on particles below the $p_T$ cut or with fewer than 3 layers inside the cut, and our times cover one tenth of an event and exclude reading and candidate building. Real trackers (ACTS, and the experiments' own) use combinatorial Kalman filters, and graph neural networks (Exa.TrkX) are the main machine-learning approach; nobody runs a MILP per event.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/ilp_track_finding/main.py
"""Main loop: real TrackML hits → candidate segments and triplets → ILP → tensor PDLP + repair, Hungarian baseline."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def near_ties(x: torch.Tensor, trip: torch.Tensor, seg: torch.Tensor, score: torch.Tensor, tol: float = 0.05) -> int:
    """Chosen triplets that have an unchosen rival sharing 2 of their 3 hits with a score within `tol`: places where
    another solver may legitimately pick a different triplet at (almost) the same objective."""
    hits = torch.stack([seg[trip[:, 0], 0], seg[trip[:, 0], 1], seg[trip[:, 1], 1]], 1)  # (T, 3)
    chosen = x.bool().nonzero().squeeze(1)
    if chosen.numel() == 0:
        return 0
    shared = (hits[chosen][:, None, :, None] == hits[None, :, None, :]).any(-1).sum(-1)  # (C, T) hits in common
    rival = (shared == 2) & ~x.bool()[None, :] & ((score[chosen][:, None] - score[None, :]).abs() < tol)
    return int(rival.any(1).sum())


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    cuts = tensors.Cuts(pt_min=params["pt_min"])
    event = tensors.load_event(device=dev)
    part = tensors.wedge(event, params["dphi"])
    n = part["xyz"].shape[0]
    n_part = int(torch.unique(part["particle"][part["particle"] > 0]).numel())
    print(
        f"TrackML event 1000, pixel barrel, |η| < 1, φ wedge {params['dphi']:g}°: {n} hits from {n_part} particles "
        f"+ {int((part['particle'] == 0).sum())} noise hits (device {dev.type})"
    )

    # 1. candidates and the ILP
    t0 = time.perf_counter()
    seg_all = equation.segments(part["xyz"], part["layer"], cuts)
    seg, trip, score = equation.candidates(part["xyz"], part["layer"], cuts)
    a, b, c = equation.ilp(seg, trip, score, n, cuts.lam)
    t_build = time.perf_counter() - t0
    print(
        f"candidates (p_T ≥ {cuts.pt_min:g} GeV, |z₀| ≤ {cuts.z0_max:g} mm): {seg_all.shape[0]} segments → "
        f"{trip.shape[0]} triplets with s_t > 0 on {seg.shape[0]} segments; ILP {a.shape[1]} variables × "
        f"{a.shape[0]} constraints ({a._nnz()} non-zeros), built in {t_build:.2f} s"
    )

    # 2. tensor PDLP on the LP relaxation, then rounding + greedy repair
    lp = solve.pdlp(a, b, c, max_iter=params["max_iter"], tol=10.0 ** -params["digits"], check=params["check"])
    t1 = time.perf_counter()
    x, y = solve.round_and_repair(lp["z"], seg, trip, score, n)
    t_repair = time.perf_counter() - t1
    obj = equation.objective(x, y, score, cuts.lam)
    feasible = bool(((a @ torch.cat([x, y]).double()) <= b + 1e-9).all())
    h = lp["history"]
    print(
        f"PDLP: {lp['iterations']} iterations in {lp['seconds']:.2f} s, primal {h['primal'][-1]:.1e}, "
        f"dual {h['dual'][-1]:.1e}, gap {h['gap'][-1]:.1e}; LP objective {lp['objective']:.3f} "
        f"(dual bound {lp['dual_bound']:.3f})"
    )
    frac = ((lp["z"] > 1e-3) & (lp["z"] < 1 - 1e-3)).double().mean().item()
    print(
        f"rounding + repair ({t_repair * 1e3:.0f} ms): {int(x.sum())} triplets, {int(y.sum())} segments, integer "
        f"objective {obj:.3f}, all constraints hold: {feasible}; {frac:.1%} of LP values fractional"
    )
    print(f"  certified: within {max(lp['dual_bound'] - obj, 0) / lp['dual_bound']:.2%} of the optimum (dual bound)")

    # 3. physics quality against the truth
    label = equation.tracks_from_segments(seg[y.bool()], n)
    eff, fake = equation.efficiency_fake(label, part["particle"], part["layer"], part["pt"], cuts.pt_min)
    score_ml = equation.trackml_score(label, part["particle"], part["weight"])
    reco = equation.reconstructable(part["particle"], part["layer"], part["pt"], cuts.pt_min)
    oracle = torch.where(reco[part["particle"]], part["particle"], -1)  # perfect tracks of reconstructable particles
    score_max = equation.trackml_score(oracle, part["particle"], part["weight"])
    print(
        f"tensor PDLP tracks: efficiency {eff:.1%} of {int(reco.sum())} reconstructable particles, fake rate "
        f"{fake:.1%}, TrackML score {score_ml:.3f} (perfect tracking of the reconstructable particles: {score_max:.3f})"
    )

    # 4. Hungarian baseline: exact layer-to-layer assignments, chained
    t2 = time.perf_counter()
    y_h = solve.hungarian_tracks(part["xyz"], part["layer"], seg, cuts)
    t_h = time.perf_counter() - t2
    x_h = solve.triplets_of(y_h, trip)
    obj_h = equation.objective(x_h, y_h.long(), score, cuts.lam)
    label_h = equation.tracks_from_segments(seg[y_h], n)
    eff_h, fake_h = equation.efficiency_fake(label_h, part["particle"], part["layer"], part["pt"], cuts.pt_min)
    score_h = equation.trackml_score(label_h, part["particle"], part["weight"])
    diff = int((x.bool() ^ x_h.bool()).sum())
    print(
        f"Hungarian chains ({t_h:.2f} s): objective {obj_h:.3f}, efficiency {eff_h:.1%}, fake rate {fake_h:.1%}, "
        f"TrackML score {score_h:.3f}; {diff} triplets differ from PDLP"
    )
    print(f"near-ties: {near_ties(x, trip, seg, score)} chosen triplets have a rival sharing 2 hits with |Δs| < 0.05")

    # 5. the offline benchmark (HiGHS, Gurobi, cuOpt attempt, PDLP on GPU, scipy assignment)
    bench = tensors.load_table(tensors.BENCHMARK)
    same = [r for r in bench if r["dphi_deg"] == params["dphi"]] if abs(cuts.pt_min - 0.5) < 1e-9 else []
    highs = [r for r in same if r["solver"] == "HiGHS MILP"]
    opt = highs[0]["objective"] if highs else None
    if opt is not None:
        print(
            f"exact MILP optimum (HiGHS, offline): {opt:.3f} → PDLP gap {max(opt - obj, 0) / opt:.2%}, "
            f"Hungarian gap {max(opt - obj_h, 0) / opt:.2%}"
        )
        for r in same:
            if r["solver"] != "HiGHS MILP" and r["n_diff_vs_highs"] != "":
                print(f"  {r['solver']}: {int(r['n_diff_vs_highs'])} selected triplets differ from HiGHS")
    else:
        print("no offline HiGHS optimum for these settings (benchmark: p_T,min = 0.5 GeV, Δφ ∈ 5…360°)")
    fits = plot.scaling_fits(bench)
    for name, (k, _) in fits.items():
        print(f"scaling: {name}: wall time ∝ N^{k:.2f}")
    live = dict(
        obj=obj,
        eff=eff,
        fake=fake,
        score=score_ml,
        seconds=lp["seconds"] + t_repair,
        n_hits=n,
        opt=opt,
        bound=lp["dual_bound"],
        obj_h=obj_h,
        eff_h=eff_h,
        fake_h=fake_h,
        score_h=score_h,
        seconds_h=t_h,
    )

    return plot.animate(
        part,
        seg_all,
        seg,
        trip,
        score,
        lp,
        label_h,
        live,
        bench,
        cuts,
        params["dphi"],
        out_dir / "ilp_track_finding.gif",
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/ilp_track_finding/equation.py
"""Track finding as an integer linear program, written as tensor operations (shapes in comments).

Hits h on layer l; candidate segments e = (a → b) join hits on adjacent layers; triplets t = (e₁, e₂) share the
middle hit. With binary x_t (triplet chosen) and y_e (segment chosen) the ILP is

    max  Σ_t s_t x_t − λ Σ_e y_e
    s.t. Σ_{e out of h} y_e ≤ 1,   Σ_{e into h} y_e ≤ 1   (each hit on at most one track: set packing)
         Σ_{t: e₁(t)=e} x_t ≤ y_e,  Σ_{t: e₂(t)=e} x_t ≤ y_e  (continuity: a triplet needs both of its segments,
                                                            and a segment continues into one triplet per side)
         x, y ∈ {0, 1}             (LP relaxation: 0 ≤ x, y ≤ 1)

i.e. max cᵀz s.t. A z ≤ b, 0 ≤ z ≤ 1 with z = (x, y). The matrix A is built sparse; its products are summed over
the variable index. Also here: the TrackML score, tracking efficiency and fake rate.
"""

import torch

from .tensors import Cuts


def wrap(angle: torch.Tensor) -> torch.Tensor:
    """Angle difference mapped to (−π, π]."""
    return torch.remainder(angle + torch.pi, 2 * torch.pi) - torch.pi


def segments(xyz: torch.Tensor, layer: torch.Tensor, cuts: Cuts) -> torch.Tensor:
    """Candidate segments (E, 2): (inner hit, outer hit) on adjacent layers that a track with p_T ≥ p_T,min from
    the beam line could join. Δφ ≤ asin(r_b/2R) − asin(r_a/2R) + tol and |z₀| ≤ z₀,max for every pair at once."""
    r = xyz[:, :2].norm(dim=-1)  # (N,)
    phi = torch.atan2(xyz[:, 1], xyz[:, 0])
    big_r = cuts.radius_min()
    out = []
    for lay in range(int(layer.max()) if layer.numel() else 0):  # 3 adjacent layer pairs
        ia = (layer == lay).nonzero().squeeze(1)  # (Na,)
        ib = (layer == lay + 1).nonzero().squeeze(1)  # (Nb,)
        ra, rb = r[ia][:, None], r[ib][None, :]  # (Na, 1), (1, Nb)
        dphi = wrap(phi[ib][None, :] - phi[ia][:, None]).abs()  # (Na, Nb)
        dphi_max = torch.asin((rb / (2 * big_r)).clamp(max=1)) - torch.asin((ra / (2 * big_r)).clamp(max=1))
        z0 = xyz[ia, 2][:, None] - ra * (xyz[ib, 2][None, :] - xyz[ia, 2][:, None]) / (rb - ra)  # (Na, Nb) mm
        ok = (dphi <= dphi_max + cuts.dphi_tol) & (z0.abs() <= cuts.z0_max)
        pa, pb = ok.nonzero().unbind(1)
        out.append(torch.stack([ia[pa], ib[pb]], 1))
    return torch.cat(out) if out else torch.zeros(0, 2, dtype=torch.long, device=xyz.device)


def doublet_curvature(xyz: torch.Tensor, seg: torch.Tensor) -> torch.Tensor:
    """Signed curvature κ (1/mm) of the circle through the beam line (0, 0) and both hits of each segment: (E,).
    κ = 2 (a × b) / (|a| |b| |b − a|) in the transverse plane."""
    a, b = xyz[seg[:, 0], :2], xyz[seg[:, 1], :2]  # (E, 2)
    cross = a[:, 0] * b[:, 1] - a[:, 1] * b[:, 0]
    return 2 * cross / (a.norm(dim=-1) * b.norm(dim=-1) * (b - a).norm(dim=-1))


def polar_angle(xyz: torch.Tensor, seg: torch.Tensor) -> torch.Tensor:
    """Polar angle θ of each segment in the r–z plane: (E,)."""
    a, b = xyz[seg[:, 0]], xyz[seg[:, 1]]
    return torch.atan2(b[:, :2].norm(dim=-1) - a[:, :2].norm(dim=-1), b[:, 2] - a[:, 2])


def z0(xyz: torch.Tensor, seg: torch.Tensor) -> torch.Tensor:
    """Where each segment, extended in r–z, crosses the beam line: (E,) mm."""
    a, b = xyz[seg[:, 0]], xyz[seg[:, 1]]
    ra, rb = a[:, :2].norm(dim=-1), b[:, :2].norm(dim=-1)
    return a[:, 2] - ra * (b[:, 2] - a[:, 2]) / (rb - ra)


def join(seg: torch.Tensor, n_hits: int) -> torch.Tensor:
    """All pairs (e₁, e₂) with outer hit of e₁ = inner hit of e₂: (T, 2) segment indices, no Python loop."""
    order = torch.argsort(seg[:, 0])  # segments sorted by inner hit
    count = torch.bincount(seg[:, 0], minlength=n_hits)  # (N,) segments leaving each hit
    start = torch.cumsum(count, 0) - count
    k = count[seg[:, 1]]  # (E,) continuations of each segment
    e1 = torch.repeat_interleave(torch.arange(seg.shape[0], device=seg.device), k)  # (T,)
    within = torch.arange(e1.numel(), device=seg.device) - torch.repeat_interleave(torch.cumsum(k, 0) - k, k)
    e2 = order[start[seg[e1, 1]] + within]
    return torch.stack([e1, e2], 1)


def triplet_score(xyz: torch.Tensor, seg: torch.Tensor, trip: torch.Tensor, cuts: Cuts) -> torch.Tensor:
    """s_t = 1 − (Δκ/σ_κ)² − (Δθ/σ_θ)²: a helix has the same curvature and polar angle on both halves. (T,)"""
    kappa, theta = doublet_curvature(xyz, seg), polar_angle(xyz, seg)  # (E,)
    dk = kappa[trip[:, 1]] - kappa[trip[:, 0]]
    dt = theta[trip[:, 1]] - theta[trip[:, 0]]
    return 1 - (dk / cuts.sigma_kappa) ** 2 - (dt / cuts.sigma_theta) ** 2


def triplets(xyz: torch.Tensor, seg: torch.Tensor, cuts: Cuts) -> tuple[torch.Tensor, torch.Tensor]:
    """Candidate triplets (T, 2) with positive score s_t (T,)."""
    trip = join(seg, xyz.shape[0])
    score = triplet_score(xyz, seg, trip, cuts)
    keep = score > 0
    return trip[keep], score[keep]


def candidates(xyz: torch.Tensor, layer: torch.Tensor, cuts: Cuts):
    """Segments (E, 2), triplets (T, 2) and scores (T,), keeping only the segments that belong to some triplet."""
    seg = segments(xyz, layer, cuts)
    trip, score = triplets(xyz, seg, cuts)
    used = torch.zeros(seg.shape[0], dtype=torch.bool, device=seg.device)
    used[trip.reshape(-1)] = True
    new_index = torch.cumsum(used.long(), 0) - 1  # old segment index → new
    return seg[used], new_index[trip], score


def ilp(seg: torch.Tensor, trip: torch.Tensor, score: torch.Tensor, n_hits: int, lam: float):
    """Sparse A (M, V), b (M,), c (V,) of  max cᵀz  s.t.  A z ≤ b,  0 ≤ z ≤ 1,  z = (x (T,), y (E,)).

    Rows: N out-degree rows, N in-degree rows, then for every segment e the continuity rows
    Σ_{t: e₁(t) = e} x_t − y_e ≤ 0 and Σ_{t: e₂(t) = e} x_t − y_e ≤ 0 (a chosen segment continues into at most one
    chosen triplet on each side; this implies x_t ≤ y_e and gives a much tighter LP): M ≤ 2N + 2E
    (empty rows dropped)."""
    n_t, n_e, dev = trip.shape[0], seg.shape[0], seg.device
    e, t = torch.arange(n_e, device=dev), torch.arange(n_t, device=dev)
    two_n = 2 * n_hits
    rows = torch.cat(
        [seg[:, 0], n_hits + seg[:, 1], two_n + trip[:, 0], two_n + n_e + trip[:, 1], two_n + e, two_n + n_e + e]
    )
    cols = torch.cat([n_t + e, n_t + e, t, t, n_t + e, n_t + e])
    vals = torch.cat([torch.ones(2 * n_e + 2 * n_t), -torch.ones(2 * n_e)]).to(dev, torch.float64)
    m, v = two_n + 2 * n_e, n_t + n_e
    b = torch.cat([torch.ones(two_n), torch.zeros(2 * n_e)]).to(dev, torch.float64)
    keep = torch.zeros(m, dtype=torch.bool, device=dev)
    keep[rows] = True  # drop empty rows (hits without any candidate segment)
    b, rows = b[keep], (torch.cumsum(keep.long(), 0) - 1)[rows]
    a = torch.sparse_coo_tensor(torch.stack([rows, cols]), vals, (int(keep.sum()), v)).coalesce()
    c = torch.cat([score.double(), torch.full((n_e,), -lam, dtype=torch.float64, device=dev)])
    return a, b, c


def objective(x: torch.Tensor, y: torch.Tensor, score: torch.Tensor, lam: float) -> float:
    """Σ s_t x_t − λ Σ y_e."""
    return (score.double() @ x.double() - lam * y.double().sum()).item()


def tracks_from_segments(chosen: torch.Tensor, n_hits: int) -> torch.Tensor:
    """Track label per hit (N,) from the chosen segments (K, 2), −1 for unused hits. Every hit has at most one
    segment in and one out, so tracks are chains; each hit gets the id of its chain's first hit (pointer jumping)."""
    dev = chosen.device
    parent = torch.arange(n_hits, device=dev)
    parent[chosen[:, 1]] = chosen[:, 0]  # each hit points to its predecessor
    for _ in range(max(1, n_hits.bit_length())):  # log₂ N doublings reach the chain root
        parent = parent[parent]
    label = torch.full((n_hits,), -1, dtype=torch.long, device=dev)
    used = torch.zeros(n_hits, dtype=torch.bool, device=dev)
    used[chosen.reshape(-1)] = True
    label[used] = parent[used]
    return label


def _pairs(label: torch.Tensor, particle: torch.Tensor):
    """Unique (track, particle) pairs with their hit counts; −1 labels become one singleton track per hit."""
    n = label.numel()
    lab = torch.where(label >= 0, label, n + torch.arange(n, device=label.device))
    key = lab * (int(particle.max()) + 1) + particle
    uniq, inv, cnt = torch.unique(key, return_inverse=True, return_counts=True)
    return lab, uniq, inv, cnt, int(particle.max()) + 1


def trackml_score(label: torch.Tensor, particle: torch.Tensor, weight: torch.Tensor) -> float:
    """Official TrackML score: a (track, particle) pair counts when more than half of the track's hits belong to
    the particle AND more than half of the particle's hits are on the track; the score is the summed hit weight of
    those intersections, normalised so that the truth scores 1. Unassigned hits (−1) are singleton tracks."""
    lab, uniq, inv, cnt, n_p = _pairs(label, particle)
    track_of, part_of = uniq // n_p, uniq % n_p
    track_size = torch.bincount(lab)[track_of]
    part_size = torch.bincount(particle, minlength=n_p)[part_of]
    good = (2 * cnt > track_size) & (2 * cnt > part_size) & (part_of > 0)
    w = torch.zeros(uniq.numel(), dtype=weight.dtype, device=weight.device).index_add_(0, inv, weight)
    return (w[good].sum() / weight.sum()).item()


def reconstructable(particle, layer, pt, pt_min: float) -> torch.Tensor:
    """Particles (P,) with p_T ≥ p_T,min and hits on at least 3 distinct layers (index 0 = noise is never)."""
    n_p = int(particle.max()) + 1
    layers_hit = torch.bincount(torch.unique(particle * 16 + layer) // 16, minlength=n_p)  # distinct layers
    pt_p = torch.zeros(n_p, dtype=pt.dtype, device=pt.device).scatter_reduce(0, particle, pt, "amax")
    reco = (layers_hit >= 3) & (pt_p >= pt_min)
    reco[0] = False
    return reco


def efficiency_fake(label, particle, layer, pt, pt_min: float) -> tuple[float, float]:
    """Efficiency: the fraction of reconstructable particles (p_T ≥ p_T,min, hits on ≥ 3 layers) with a track that
    holds ≥ 3 of their hits. Fake rate: the fraction of tracks (≥ 3 hits) with no particle owning ≥ 3 of its hits."""
    n_p = int(particle.max()) + 1
    reco = reconstructable(particle, layer, pt, pt_min)
    lab, uniq, inv, cnt, _ = _pairs(label, particle)
    track_of, part_of = uniq // n_p, uniq % n_p
    matched = (cnt >= 3) & (part_of > 0)
    found = torch.zeros(n_p, dtype=torch.bool, device=label.device)
    found[part_of[matched]] = True
    eff = (found & reco).sum().item() / max(1, reco.sum().item())
    sizes = torch.bincount(lab)
    tracks = (sizes >= 3).nonzero().squeeze(1)
    good_track = torch.zeros(sizes.numel(), dtype=torch.bool, device=label.device)
    good_track[track_of[matched]] = True
    fake = (~good_track[tracks]).sum().item() / max(1, tracks.numel())
    return eff, fake


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/ilp_track_finding/tensors.py
"""Build the concrete tensors: the real TrackML hits, the φ wedge that sets the problem size, toy events and cuts.

data/trackml_event1000_barrel.csv is one simulated proton–proton event of the TrackML Particle Tracking Challenge
(CERN / Kaggle 2018, ⟨μ⟩ = 200 pile-up collisions at HL-LHC conditions), training event 1000, cut to the four
pixel-barrel layers (volume 8) and |η| < 1: 11 519 hits from 2 452 particles plus 6 % noise hits. Source: Zenodo
record 4730167 (doi:10.5281/zenodo.4730167), licence CC BY 4.0. The other CSVs are the offline solver benchmark and
the cited TrackML state of the art.
"""

import math
from dataclasses import dataclass
from pathlib import Path

import torch

HERE = Path(__file__).parent / "data"
DATA = HERE / "trackml_event1000_barrel.csv"
PARTICLES = HERE / "trackml_particles.csv"
BENCHMARK = HERE / "solver_benchmark.csv"
SOTA = HERE / "trackml_sota.csv"
DTYPE = torch.float64
B_FIELD = 2.0  # T, solenoid field of the TrackML detector
LAYER_R = (32.3, 72.1, 116.1, 172.1)  # mm, mean radius of the four pixel-barrel layers (volume 8)
N_LAYERS = len(LAYER_R)


@dataclass
class Cuts:
    """Geometric cuts for candidate segments, the triplet score widths and the cost of an unused segment."""

    pt_min: float = 1.0  # GeV: tracks softer than this curl too much to pass the Δφ cut
    z0_max: float = 150.0  # mm: the segment must point back to the luminous region on the beam line
    dphi_tol: float = 0.004  # rad: extra Δφ allowed for multiple scattering and module tilt
    sigma_kappa: float = 1.2e-4  # 1/mm: width of the curvature mismatch Δκ between the two halves of a triplet
    sigma_theta: float = 0.006  # rad: width of the polar-angle mismatch Δθ
    lam: float = 0.1  # cost of a selected segment (only segments that carry a good triplet pay for themselves)

    def radius_min(self) -> float:
        """Smallest helix radius allowed, in mm: R = p_T / (0.3 B) m with p_T in GeV and B in T."""
        return self.pt_min / (0.3 * B_FIELD) * 1000.0


def _read_csv(path: Path) -> tuple[list[str], list[list[str]]]:
    lines = [ln for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip() and ln[0] != "#"]
    return lines[0].split(","), [ln.split(",") for ln in lines[1:]]


def load_table(path: Path) -> list[dict]:
    """A small CSV (lines starting with # are comments) as a list of dicts, numbers converted where possible."""
    header, rows = _read_csv(path)

    def num(v: str):
        try:
            return float(v)
        except ValueError:
            return v

    return [{h: num(v) for h, v in zip(header, row)} for row in rows]


def load_event(path: Path = DATA, particles: Path = PARTICLES, device=None) -> dict:
    """Read the hits: xyz (N, 3) mm, layer (N,), particle (N,) (0 = noise), weight (N,), pt (N,) GeV (0 for noise)."""
    header, rows = _read_csv(path)
    t = torch.tensor([[float(v) for v in r] for r in rows], dtype=DTYPE)
    col = {h: t[:, i] for i, h in enumerate(header)}
    p_header, p_rows = _read_csv(particles)
    p = torch.tensor([[float(v) for v in r] for r in p_rows], dtype=DTYPE)
    pt_of = torch.zeros(int(p[:, 0].max()) + 1, dtype=DTYPE)
    pt_of[p[:, 0].long()] = p[:, p_header.index("pt")]
    particle = col["particle"].long()
    event = {
        "xyz": torch.stack([col["x"], col["y"], col["z"]], -1),
        "layer": col["layer"].long(),
        "particle": particle,
        "weight": col["weight"],
        "pt": pt_of[particle],
    }
    return {k: v.to(device) for k, v in event.items()}


def wedge(event: dict, dphi_deg: float, phi0_deg: float = 0.0) -> dict:
    """Keep the hits with azimuth φ in [φ₀, φ₀ + Δφ): the problem size grows linearly with Δφ (360° = all hits)."""
    phi = torch.rad2deg(torch.atan2(event["xyz"][:, 1], event["xyz"][:, 0]))  # (N,) in (−180°, 180°]
    keep = torch.remainder(phi - phi0_deg, 360.0) < dphi_deg
    return {k: v[keep] for k, v in event.items()}


def helix_hits(pt, phi0, eta, z0, charge, radii) -> torch.Tensor:
    """Where a helix from the beam line (x = y = 0, z = z0) crosses cylinders of radius r: (P, L, 3) in mm.

    In a solenoid B ẑ the transverse path is a circle of radius R = p_T/(0.3 B) through the origin; at radius r it
    has turned by asin(r/2R), and the arc length s = 2R asin(r/2R) sets z = z0 + s sinh η.
    """
    big_r = (pt / (0.3 * B_FIELD) * 1000.0)[:, None]  # (P, 1) mm
    r = radii[None, :]  # (1, L)
    turn = torch.asin(r / (2 * big_r))  # (P, L)
    phi = phi0[:, None] - charge[:, None] * turn
    z = z0[:, None] + 2 * big_r * turn * torch.sinh(eta)[:, None]
    return torch.stack([r * torch.cos(phi), r * torch.sin(phi), z], -1)


def toy_event(n_particles: int, noise_fraction: float = 0.0, seed: int = 0, device=None) -> dict:
    """A TrackML-like toy: helices with p_T in 1.5–10 GeV, evenly spread in φ (well separated), on the 4 barrel
    layers, plus uniform noise hits. Same keys as load_event."""
    gen = torch.Generator().manual_seed(seed)
    radii = torch.tensor(LAYER_R, dtype=DTYPE)
    pt = 1.5 + 8.5 * torch.rand(n_particles, generator=gen, dtype=DTYPE)
    phi0 = (torch.arange(n_particles, dtype=DTYPE) + 0.3 * torch.rand(n_particles, generator=gen, dtype=DTYPE)) * (
        2 * math.pi / n_particles
    )
    eta = 1.6 * torch.rand(n_particles, generator=gen, dtype=DTYPE) - 0.8
    z0 = 40.0 * torch.randn(n_particles, generator=gen, dtype=DTYPE).clamp(-2.5, 2.5)
    charge = torch.where(torch.rand(n_particles, generator=gen) < 0.5, -1.0, 1.0).to(DTYPE)
    xyz = helix_hits(pt, phi0, eta, z0, charge, radii).reshape(-1, 3)  # (P·L, 3)
    layer = torch.arange(N_LAYERS).repeat(n_particles)
    particle = torch.arange(1, n_particles + 1).repeat_interleave(N_LAYERS)
    n_noise = int(round(noise_fraction * xyz.shape[0]))
    if n_noise:
        nl = torch.randint(0, N_LAYERS, (n_noise,), generator=gen)
        ph = 2 * math.pi * torch.rand(n_noise, generator=gen, dtype=DTYPE)
        zz = radii[nl] * torch.sinh(1.6 * torch.rand(n_noise, generator=gen, dtype=DTYPE) - 0.8)
        noise = torch.stack([radii[nl] * torch.cos(ph), radii[nl] * torch.sin(ph), zz], -1)
        xyz, layer = torch.cat([xyz, noise]), torch.cat([layer, nl])
        particle = torch.cat([particle, torch.zeros(n_noise, dtype=torch.long)])
    pts = torch.cat([torch.zeros(1, dtype=DTYPE), pt])[particle]
    weight = (particle > 0).to(DTYPE)
    event = {"xyz": xyz, "layer": layer, "particle": particle, "weight": weight / weight.sum(), "pt": pts}
    return {k: v.to(device) for k, v in event.items()}


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/ilp_track_finding/solve.py
"""The solvers: a tensor PDLP for the LP relaxation, rounding + greedy repair to integers, and Hungarian matching.

PDLP (Applegate et al. 2021) is the primal–dual hybrid gradient method for LP with diagonal (Ruiz) preconditioning,
averaging, adaptive restarts and a primal weight; the same method runs inside Google OR-Tools and NVIDIA cuOpt.
Each iteration costs two sparse matrix–vector products, so it runs unchanged on CPU or CUDA.
"""

import math
import time
import warnings

import torch

from problems._common.progress import report

from . import equation
from .tensors import Cuts


def _ruiz(idx: torch.Tensor, val: torch.Tensor, m: int, n: int, iters: int = 12):
    """Row and column scalings D_r (M,), D_c (V,) that equilibrate |D_r A D_c| to max 1 in every row and column."""
    dr = torch.ones(m, dtype=val.dtype, device=val.device)
    dc = torch.ones(n, dtype=val.dtype, device=val.device)
    for _ in range(iters):
        v = (val * dr[idx[0]] * dc[idx[1]]).abs()
        rmax = torch.zeros_like(dr).scatter_reduce(0, idx[0], v, "amax")
        cmax = torch.zeros_like(dc).scatter_reduce(0, idx[1], v, "amax")
        dr = dr / torch.where(rmax > 0, rmax.sqrt(), torch.ones_like(rmax))
        dc = dc / torch.where(cmax > 0, cmax.sqrt(), torch.ones_like(cmax))
    return dr, dc


def _norm2(a: torch.Tensor, at: torch.Tensor, iters: int = 40) -> float:
    """Largest singular value of A by power iteration on AᵀA."""
    v = torch.ones(a.shape[1], dtype=torch.float64, device=a.device) / math.sqrt(a.shape[1])
    s = 1.0
    for _ in range(iters):
        w = at @ (a @ v)
        s = w.norm().item()
        v = w / max(s, 1e-300)
    return math.sqrt(s)


def kkt(a, at, b, c, z, y) -> dict:
    """Optimality measures of (z, y) for  min qᵀz, q = −c,  A z ≤ b,  0 ≤ z ≤ 1,  y ≥ 0  (unscaled).

    Reduced cost λ = q + Aᵀy. Dual objective D = −bᵀy + Σ min(λ, 0) is a lower bound for every y ≥ 0 because the
    box is finite. primal: ‖(Az − b)₊‖; dual: ‖λ⁺z + λ⁻(1 − z)‖ (reduced costs that disagree with the bounds);
    gap: |P − D|. All three are 0 exactly at an optimum (complementary slackness)."""
    q = -c
    lam = q + at @ y  # (V,)
    p_obj = (q @ z).item()
    d_obj = (-b @ y + lam.clamp(max=0).sum()).item()
    primal = (a @ z - b).clamp(min=0).norm().item() / (1 + b.norm().item())
    dual = (lam.clamp(min=0) * z + (-lam).clamp(min=0) * (1 - z)).norm().item() / (1 + c.norm().item())
    gap = abs(p_obj - d_obj) / (1 + abs(p_obj) + abs(d_obj))
    return {"primal": primal, "dual": dual, "gap": gap, "objective": -p_obj, "dual_bound": -d_obj}


def pdlp(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor, max_iter: int = 5000, tol: float = 1e-5, check: int = 64):
    """Solve  max cᵀz  s.t.  A z ≤ b,  0 ≤ z ≤ 1  by restarted, preconditioned PDHG (PDLP).

    Scaled problem Â = D_r A D_c, ẑ = z/D_c with box [0, 1/D_c]. One iteration:
        ẑ⁺ = clip(ẑ − τ(q̂ + Âᵀŷ), 0, û),     ŷ⁺ = max(0, ŷ + σ(Â(2ẑ⁺ − ẑ) − b̂)),    τ = η/ω, σ = ηω, η‖Â‖ < 1.
    Every `check` iterations the current and averaged iterates are scored by their KKT error; restart to the better
    one on sufficient (×0.2) or necessary (×0.8, no longer improving) decay, then update the primal weight ω.
    Returns z, y, objective, dual_bound, history (iteration, primal, dual, gap) and snapshots of z."""
    t0 = time.perf_counter()
    warnings.filterwarnings("ignore", message="Sparse (CSR tensor support|invariant checks)")
    a = a.coalesce()
    m, n = a.shape
    idx, val = a.indices(), a.values().double()
    dr, dc = _ruiz(idx, val, m, n)
    a_hat = torch.sparse_coo_tensor(idx, val * dr[idx[0]] * dc[idx[1]], (m, n)).coalesce().to_sparse_csr()
    at_hat = a_hat.t().to_sparse_csr()
    a_csr, at_csr = a.to_sparse_csr(), a.t().coalesce().to_sparse_csr()
    q_hat, b_hat, u_hat = -c * dc, b * dr, 1.0 / dc
    eta = 0.95 / _norm2(a_hat, at_hat)
    bn, qn = b_hat.norm().item(), q_hat.norm().item()
    omega = qn / bn if bn > 1e-12 and qn > 1e-12 else 1.0
    z = torch.zeros(n, dtype=torch.float64, device=c.device)
    y = torch.zeros(m, dtype=torch.float64, device=c.device)
    z_sum, y_sum, n_avg = torch.zeros_like(z), torch.zeros_like(y), 0
    z_last, y_last, kkt_last, kkt_prev = z.clone(), y.clone(), math.inf, math.inf
    history, snaps = [], []

    def score(zh, yh):
        m_ = kkt(a_csr, at_csr, b, c, dc * zh, dr * yh)
        return m_, math.sqrt(omega**2 * m_["primal"] ** 2 + m_["dual"] ** 2 / omega**2 + m_["gap"] ** 2), m_

    best = None
    for k in range(1, max_iter + 1):
        tau, sigma = eta / omega, eta * omega
        z_new = torch.minimum(torch.clamp(z - tau * (q_hat + at_hat @ y), min=0), u_hat)
        y = torch.clamp(y + sigma * (a_hat @ (2 * z_new - z) - b_hat), min=0)
        z = z_new
        z_sum += z
        y_sum += y
        n_avg += 1
        if k % check and k != max_iter:
            continue
        m_cur, k_cur, _ = score(z, y)
        m_avg, k_avg, _ = score(z_sum / n_avg, y_sum / n_avg)
        use_avg = k_avg < k_cur
        cand_z, cand_y = (z_sum / n_avg, y_sum / n_avg) if use_avg else (z, y)
        k_cand, m_cand = (k_avg, m_avg) if use_avg else (k_cur, m_cur)
        best = (dc * cand_z, dr * cand_y, m_cand)
        history.append((k, m_cand["primal"], m_cand["dual"], m_cand["gap"], m_cand["objective"]))
        snaps.append((k, (dc * cand_z).clamp(0, 1).clone()))
        report(k, max_iter, "PDLP iterations")
        if max(m_cand["primal"], m_cand["dual"], m_cand["gap"]) < tol:
            break
        if k_cand <= 0.2 * kkt_last or (k_cand <= 0.8 * kkt_last and k_cand > kkt_prev) or n_avg >= 0.36 * k:
            dz, dy = (cand_z - z_last).norm().item(), (cand_y - y_last).norm().item()
            if dz > 1e-10 and dy > 1e-10:  # primal weight: balance the distance travelled in z and in y
                omega = math.exp(0.5 * math.log(dy / dz) + 0.5 * math.log(omega))
            z, y = cand_z.clone(), cand_y.clone()
            z_last, y_last, kkt_last = z.clone(), y.clone(), k_cand
            z_sum, y_sum, n_avg = torch.zeros_like(z), torch.zeros_like(y), 0
        kkt_prev = k_cand
    z_u, y_u, m_fin = best
    hist = torch.tensor(history, dtype=torch.float64)
    return {
        "z": z_u.clamp(0, 1),
        "y": y_u,
        "objective": m_fin["objective"],
        "dual_bound": m_fin["dual_bound"],
        "iterations": k,
        "history": {"iter": hist[:, 0], "primal": hist[:, 1], "dual": hist[:, 2], "gap": hist[:, 3], "obj": hist[:, 4]},
        "snaps": snaps,
        "seconds": time.perf_counter() - t0,
    }


def round_and_repair(z, seg, trip, score, n_hits: int, lam: float = Cuts.lam):
    """Integer (x (T,), y (E,)) from an LP point z = (x, y): rounding then a greedy conflict repair, in parallel.

    Segments are ranked by LP value y_e (ties: best triplet score). Each round accepts every segment that ranks
    first among the undecided segments leaving its inner hit AND entering its outer hit, then discards segments
    that clash with an accepted one; this is exactly sequential greedy, done as tensor rounds. Triplets are kept
    when both segments are accepted; segments that carry no kept triplet are dropped (they only cost λ), and chain
    ends whose last triplet scores below λ (a lone triplet below 2λ) are trimmed."""
    n_t, n_e, dev = trip.shape[0], seg.shape[0], seg.device
    y_lp = z[n_t:]
    best_s = torch.zeros(n_e, dtype=torch.float64, device=dev)
    best_s.scatter_reduce_(0, trip.reshape(-1), score.double().repeat_interleave(2), "amax")
    useful = torch.zeros(n_e, dtype=torch.bool, device=dev)
    useful[trip.reshape(-1)] = True
    order = torch.argsort(y_lp + 1e-3 * best_s, stable=True)
    rank = torch.empty(n_e, dtype=torch.float64, device=dev)
    rank[order] = torch.arange(n_e, dtype=torch.float64, device=dev)  # unique priorities
    undecided, chosen = useful.clone(), torch.zeros(n_e, dtype=torch.bool, device=dev)
    busy_out = torch.zeros(n_hits, dtype=torch.bool, device=dev)
    busy_in = torch.zeros(n_hits, dtype=torch.bool, device=dev)
    while undecided.any():
        p = torch.where(undecided, rank, torch.full_like(rank, -1.0))
        top_out = torch.full((n_hits,), -1.0, dtype=torch.float64, device=dev).scatter_reduce(0, seg[:, 0], p, "amax")
        top_in = torch.full((n_hits,), -1.0, dtype=torch.float64, device=dev).scatter_reduce(0, seg[:, 1], p, "amax")
        win = undecided & (p == top_out[seg[:, 0]]) & (p == top_in[seg[:, 1]])
        chosen |= win
        busy_out[seg[win, 0]] = True
        busy_in[seg[win, 1]] = True
        undecided &= ~win & ~busy_out[seg[:, 0]] & ~busy_in[seg[:, 1]]
    while True:  # trim chain ends whose last triplet is worth less than the segments it needs
        x = chosen[trip[:, 0]] & chosen[trip[:, 1]]
        y = torch.zeros(n_e, dtype=torch.bool, device=dev)
        y[trip[x].reshape(-1)] = True
        as_first = torch.zeros(n_e, dtype=torch.bool, device=dev)
        as_second = torch.zeros(n_e, dtype=torch.bool, device=dev)
        as_first[trip[x, 0]] = True
        as_second[trip[x, 1]] = True
        weak = x & (score < lam)
        drop_out = weak & ~as_first[trip[:, 1]]  # outer segment ends the chain
        drop_in = weak & ~as_second[trip[:, 0]]  # inner segment starts the chain
        alone = x & ~as_first[trip[:, 1]] & ~as_second[trip[:, 0]] & (score < 2 * lam)  # a lone triplet pays 2λ
        drop = torch.zeros(n_e, dtype=torch.bool, device=dev)
        drop[trip[drop_out, 1]] = True
        drop[trip[drop_in & ~drop_out, 0]] = True
        drop[trip[alone].reshape(-1)] = True
        if not drop.any():
            return x.long(), y.long()
        chosen = y & ~drop


def hungarian(cost: torch.Tensor) -> torch.Tensor:
    """Exact minimum-cost assignment (Kuhn–Munkres with potentials, O(n² m)): column of each row, (n,), n ≤ m.

    Rows are added one at a time; a shortest augmenting path (Dijkstra on reduced costs c_ij − u_i − v_j ≥ 0) is
    grown over all columns at once with vector operations, then the potentials u, v are updated by the step δ."""
    n, m = cost.shape
    if n > m:
        col_of_row_t = hungarian(cost.t())  # (m,) row of each column
        out = torch.empty(n, dtype=torch.long, device=cost.device).fill_(-1)
        out[col_of_row_t] = torch.arange(m, device=cost.device)
        return out
    c = cost.double()
    inf = torch.tensor(float("inf"), dtype=torch.float64, device=c.device)
    u = torch.zeros(n + 1, dtype=torch.float64, device=c.device)
    v = torch.zeros(m + 1, dtype=torch.float64, device=c.device)
    p = torch.zeros(m + 1, dtype=torch.long, device=c.device)  # p[j]: row matched to column j (1-based, 0 = free)
    way = torch.zeros(m + 1, dtype=torch.long, device=c.device)
    for i in range(1, n + 1):
        p[0] = i
        j0 = 0
        minv = torch.full((m + 1,), float("inf"), dtype=torch.float64, device=c.device)
        used = torch.zeros(m + 1, dtype=torch.bool, device=c.device)
        while True:
            used[j0] = True
            i0 = int(p[j0])
            cur = c[i0 - 1] - u[i0] - v[1:]  # (m,) reduced costs from row i0
            better = ~used[1:] & (cur < minv[1:])
            minv[1:] = torch.where(better, cur, minv[1:])
            way[1:] = torch.where(better, torch.tensor(j0, device=c.device), way[1:])
            masked = torch.where(used[1:], inf, minv[1:])
            j1 = int(torch.argmin(masked)) + 1
            delta = masked[j1 - 1]
            u.index_add_(0, p[used], delta.expand(int(used.sum())))
            v = torch.where(used, v - delta, v)
            minv = torch.where(used, minv, minv - delta)
            j0 = j1
            if int(p[j0]) == 0:
                break
        while j0:  # augment along the path
            j1 = int(way[j0])
            p[j0] = p[j1]
            j0 = j1
    col = torch.empty(n, dtype=torch.long, device=c.device)
    matched = p[1:] > 0
    col[p[1:][matched] - 1] = torch.arange(m, device=c.device)[matched]
    return col


def hungarian_tracks(xyz, layer, seg, cuts: Cuts, assign=None) -> torch.Tensor:
    """Baseline: layer-to-layer linear assignment problems, chained into tracks. Returns a bool mask (E,).

    Each step matches hits on layer l to at most one hit on layer l+1 (a 'no match' column costs 1). The middle
    layer pair goes first with cost 1 − (best triplet score the segment could join); then the inner pair, whose cost
    is 1 − s_t of the triplet with the already chosen continuation, then the outer pair the same way (curvature and
    angle mismatch over three layers, but decided one layer pair at a time). Chains shorter than 3 hits are dropped.
    `assign` swaps in another exact assignment solver (e.g. scipy's linear_sum_assignment, offline)."""
    assign = assign or hungarian
    n, n_e, dev = xyz.shape[0], seg.shape[0], xyz.device
    trip_all = equation.join(seg, n)
    s_all = equation.triplet_score(xyz, seg, trip_all, cuts)
    trip_all, s_all = trip_all[s_all > 0], s_all[s_all > 0]
    support = torch.zeros(n_e, dtype=torch.float64, device=dev)
    support.scatter_reduce_(0, trip_all.reshape(-1), s_all.repeat_interleave(2), "amax")
    pred = torch.full((n,), -1, dtype=torch.long, device=dev)  # chosen segment entering each hit
    succ = torch.full((n,), -1, dtype=torch.long, device=dev)  # chosen segment leaving each hit
    chosen = torch.zeros(n_e, dtype=torch.bool, device=dev)
    n_pairs = int(layer.max()) if n else 0
    first = n_pairs // 2
    for lay in [first, *range(first - 1, -1, -1), *range(first + 1, n_pairs)]:  # middle, inwards, outwards
        es = (layer[seg[:, 0]] == lay).nonzero().squeeze(1)
        if es.numel() == 0:
            continue
        cost_e = 1 - support[es]
        if lay != first:
            cost_e = torch.ones_like(cost_e)
            mine = torch.zeros(n_e, dtype=torch.bool, device=dev)
            mine[es] = True
            pos = torch.full((n_e,), -1, dtype=torch.long, device=dev)
            pos[es] = torch.arange(es.numel(), device=dev)
            if lay < first:  # triplet (e, chosen continuation of e's outer hit)
                ok = mine[trip_all[:, 0]] & (trip_all[:, 1] == succ[seg[trip_all[:, 0], 1]])
                cost_e[pos[trip_all[ok, 0]]] = 1 - s_all[ok]
            else:  # triplet (chosen segment into e's inner hit, e)
                ok = mine[trip_all[:, 1]] & (trip_all[:, 0] == pred[seg[trip_all[:, 1], 0]])
                cost_e[pos[trip_all[ok, 1]]] = 1 - s_all[ok]
        rows, ri = torch.unique(seg[es, 0], return_inverse=True)
        cols, ci = torch.unique(seg[es, 1], return_inverse=True)
        cost = torch.full((rows.numel(), cols.numel() + rows.numel()), 10.0, dtype=torch.float64, device=dev)
        cost[:, cols.numel() :] = 1.0  # 'no match' columns
        cost[ri, ci] = cost_e.clamp(max=10.0)
        col = assign(cost)
        real = (col < cols.numel()) & (cost[torch.arange(rows.numel(), device=dev), col] < 1.0)
        key = torch.full((rows.numel(), cols.numel()), -1, dtype=torch.long, device=dev)
        key[ri, ci] = es
        picked = key[real.nonzero().squeeze(1), col[real]]
        chosen[picked] = True
        pred[seg[picked, 1]] = picked
        succ[seg[picked, 0]] = picked
    label = equation.tracks_from_segments(seg[chosen], n)
    size = torch.bincount(label[label >= 0], minlength=n)
    return chosen & (size[label[seg[:, 0]].clamp(min=0)] >= 3)


def triplets_of(chosen_seg: torch.Tensor, trip: torch.Tensor) -> torch.Tensor:
    """x_t = 1 when both segments of triplet t are chosen (T,)."""
    return (chosen_seg[trip[:, 0]] & chosen_seg[trip[:, 1]]).long()


def fit_scaling(size: torch.Tensor, seconds: torch.Tensor) -> tuple[float, float]:
    """Least squares in log–log: t ≈ C nᵏ. Returns (k, C)."""
    x, y = size.double().log(), seconds.double().log()
    design = torch.stack([x, torch.ones_like(x)], 1)
    k, logc = torch.linalg.lstsq(design, y[:, None]).solution[:, 0]
    return k.item(), math.exp(logc.item())


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/ilp_track_finding/plot.py
"""Candidate segments fade in, then the chosen tracks appear as PDLP converges; solver comparison panels below.

Top: the detector seen along the beam (x–y) and from the side (r–z), and the PDLP residuals against iteration.
Bottom: wall time against problem size for every solver (log–log, fitted exponents), the gap / efficiency / fake
rate of each solver at this size, and the TrackML score against time per event next to the cited state of the art.
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import LineCollection
from PIL import Image

from problems._common.progress import report

from . import equation, solve, tensors

GREY, INK = "#9ca3af", "#111827"
SOLVERS = {  # one colour per solver, used in every panel
    "HiGHS MILP": "#111827",
    "HiGHS LP": "#6b7280",
    "Gurobi MILP": "#dc2626",
    "tensor PDLP (CPU)": "#2563eb",
    "tensor PDLP (CUDA)": "#06b6d4",
    "Hungarian (ours; PyTorch)": "#f59e0b",
    "scipy linear_sum_assignment": "#a16207",
}
SHORT = {
    "HiGHS MILP": "HiGHS MILP",
    "HiGHS LP": "HiGHS LP",
    "Gurobi MILP": "Gurobi",
    "tensor PDLP (CPU)": "PDLP CPU",
    "tensor PDLP (CUDA)": "PDLP GPU",
    "Hungarian (ours; PyTorch)": "Hungarian",
    "scipy linear_sum_assignment": "scipy LSA",
}
SOTA_COLOUR = "#16a34a"
TRACK_COLOURS = [
    "#2563eb",
    "#dc2626",
    "#16a34a",
    "#9333ea",
    "#ea580c",
    "#0891b2",
    "#db2777",
    "#65a30d",
    "#4f46e5",
    "#b45309",
    "#0d9488",
    "#be123c",
]
N_FADE, N_PDLP, N_HOLD = 14, 44, 16
RC = {  # readable at dpi 80: titles 11 pt, axis labels and ticks 10 pt, legends and notes 9 pt
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
}


def scaling_fits(bench: list[dict]) -> dict:
    """Fitted t ≈ C Nᵏ per solver, N = hits in the wedge: on N ≥ 1000 when there are ≥ 3 such sizes, else all."""
    fits = {}
    for name in SOLVERS:
        rows = [r for r in bench if r["solver"] == name and isinstance(r["wall_s"], float)]
        large = [r for r in rows if r["n_hits"] >= 1000]  # fixed overheads dominate the small instances
        rows = large if len(large) >= 3 else rows
        if len(rows) >= 3:
            n = torch.tensor([r["n_hits"] for r in rows], dtype=torch.float64)
            t = torch.tensor([r["wall_s"] for r in rows], dtype=torch.float64)
            fits[name] = solve.fit_scaling(n, t)
    return fits


def _save_shared_palette(fig, update, frames, path: Path, fps: int = 10) -> Path:
    """Render every frame, quantise all of them with ONE palette (colours never drift), write the GIF."""
    fig.set_dpi(80)
    shots = []
    for i, f in enumerate(frames):
        update(f)
        fig.canvas.draw()
        shots.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]))
        report(i + 1, len(frames), "frames")
    plt.close(fig)
    sample = shots[:: max(1, len(shots) // 12)]
    strip = Image.new("RGB", (sample[0].width, sample[0].height * len(sample)))
    for k, im in enumerate(sample):
        strip.paste(im, (0, k * im.height))
    palette = strip.quantize(colors=255, method=Image.Quantize.MEDIANCUT)
    frames_p = [im.quantize(palette=palette, dither=Image.Dither.NONE) for im in shots]
    tmp = path.with_name(f".{path.stem}.tmp{path.suffix}")
    frames_p[0].save(tmp, save_all=True, append_images=frames_p[1:], duration=int(1000 / fps), loop=0)
    return tmp.replace(path)


def _track_lines(xyz, seg, chosen, label):
    """x–y and r–z line segments of the chosen segments, with one colour per track."""
    s = seg[chosen]
    p, q = xyz[s[:, 0]], xyz[s[:, 1]]
    xy = np.stack([p[:, :2], q[:, :2]], 1)
    rz = np.stack(
        [np.stack([p[:, 2], np.hypot(p[:, 0], p[:, 1])], -1), np.stack([q[:, 2], np.hypot(q[:, 0], q[:, 1])], -1)], 1
    )
    colours = [TRACK_COLOURS[int(k) % len(TRACK_COLOURS)] for k in label[s[:, 0]]]
    return xy, rz, colours


def _animate(part, seg_all, seg, trip, score, lp, label_h, live, bench, cuts, dphi, path: Path) -> Path:
    xyz = part["xyz"].cpu().numpy()
    seg_c, trip_c, score_c = seg.cpu(), trip.cpu(), score.cpu()
    n = xyz.shape[0]
    r = np.hypot(xyz[:, 0], xyz[:, 1])
    hist = {k: v.cpu().numpy() for k, v in lp["history"].items()}
    snaps = lp["snaps"]
    pick = sorted(set(np.linspace(0, len(snaps) - 1, min(N_PDLP, len(snaps))).round().astype(int).tolist()))
    states = []
    for i in pick:
        x, y = solve.round_and_repair(snaps[i][1].cpu(), seg_c, trip_c, score_c, n)
        lab = equation.tracks_from_segments(seg_c[y.bool()], n)
        states.append((snaps[i][0], *_track_lines(xyz, seg_c.numpy(), y.bool().numpy(), lab.numpy()), int(x.sum())))

    fig = plt.figure(figsize=(14, 8.6))
    grid = fig.add_gridspec(
        2, 3, height_ratios=[1.1, 1], hspace=0.42, wspace=0.34, left=0.06, right=0.98, top=0.95, bottom=0.1
    )
    ax_xy, ax_rz, ax_res = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[0, 2])
    ax_sc = fig.add_subplot(grid[1, 0])
    sub = grid[1, 1].subgridspec(1, 2, width_ratios=[1, 1.3], wspace=0.08)
    ax_gap, ax_eff = fig.add_subplot(sub[0]), fig.add_subplot(sub[1])
    ax_ml = fig.add_subplot(grid[1, 2])

    # detector views (fixed limits from all hits)
    pad = 8.0
    ax_xy.set_xlim(xyz[:, 0].min() - pad, xyz[:, 0].max() + pad)
    ax_xy.set_ylim(xyz[:, 1].min() - pad, xyz[:, 1].max() + pad)
    ax_xy.set_aspect("equal", adjustable="datalim")
    ax_xy.set_xlabel("$x$ (mm)")
    ax_xy.set_ylabel("$y$ (mm)")
    ax_rz.set_xlim(xyz[:, 2].min() - 20, xyz[:, 2].max() + 20)
    ax_rz.set_ylim(0, r.max() + 15)
    ax_rz.set_xlabel("$z$ (mm)")
    ax_rz.set_ylabel("$r$ (mm)")
    noise = part["particle"].cpu().numpy() == 0
    for ax, (u, v) in ((ax_xy, (xyz[:, 0], xyz[:, 1])), (ax_rz, (xyz[:, 2], r))):
        ax.scatter(u[~noise], v[~noise], s=4, color=INK, zorder=3, lw=0)
        ax.scatter(u[noise], v[noise], s=5, color=GREY, marker="x", zorder=3, lw=0.6)
    sa = seg_all.cpu().numpy()
    if len(sa) > 20000:  # draw a fixed random sample of the candidates on very wide wedges
        sa = sa[np.random.default_rng(0).choice(len(sa), 20000, replace=False)]
    cand_xy = LineCollection(np.stack([xyz[sa[:, 0], :2], xyz[sa[:, 1], :2]], 1), colors=GREY, lw=0.3, alpha=0.0)
    cand_rz = LineCollection(
        np.stack([np.stack([xyz[sa[:, 0], 2], r[sa[:, 0]]], -1), np.stack([xyz[sa[:, 1], 2], r[sa[:, 1]]], -1)], 1),
        colors=GREY,
        lw=0.3,
        alpha=0.0,
    )
    ax_xy.add_collection(cand_xy)
    ax_rz.add_collection(cand_rz)
    trk_xy = LineCollection([], lw=1.6, zorder=4)
    trk_rz = LineCollection([], lw=1.6, zorder=4)
    ax_xy.add_collection(trk_xy)
    ax_rz.add_collection(trk_rz)
    title_xy = ax_xy.set_title("")
    ax_rz.set_title(f"side view: {len(seg_all)} segments, {trip.shape[0]} triplets")

    # residuals: axis ranges from the whole history, curves drawn progressively
    it = hist["iter"]
    floor = 1e-12
    ax_res.set_xlim(0, it.max() * 1.02)
    vals = np.concatenate([hist["primal"], hist["dual"], hist["gap"]])
    ax_res.set_yscale("log")
    ax_res.set_ylim(floor * 0.3, vals.max() * 3)
    ax_res.set_xlabel("PDLP iteration (dimensionless)")
    ax_res.set_ylabel("relative residual (dimensionless)")
    (l_p,) = ax_res.plot([], [], "o-", ms=3, color="#2563eb", lw=1.8, label="primal")
    (l_d,) = ax_res.plot([], [], "o-", ms=3, color="#dc2626", lw=1.8, label="dual")
    (l_g,) = ax_res.plot([], [], "o-", ms=3, color="#16a34a", lw=1.8, label="duality gap")
    ax_res.legend(loc="upper right")
    res_title = ax_res.set_title("")

    # scalability
    fits = scaling_fits(bench)
    sizes = [rw["n_hits"] for rw in bench]
    ax_sc.set_xscale("log")
    ax_sc.set_yscale("log")
    for name, colour in SOLVERS.items():
        rows = [rw for rw in bench if rw["solver"] == name and isinstance(rw["wall_s"], float)]
        if not rows:
            continue
        nn = np.array([rw["n_hits"] for rw in rows])
        tt = np.array([rw["wall_s"] for rw in rows])
        lab = SHORT[name] + (f" $N^{{{fits[name][0]:.2f}}}$" if name in fits else "")
        ax_sc.plot(nn, tt, "o", ms=4, color=colour, label=lab)
        if name in fits:
            k, cc = fits[name]
            g = np.geomspace(min(nn), max(nn), 20)
            ax_sc.plot(g, cc * g**k, color=colour, lw=1.2, ls="--" if "LP" in name and "PDLP" not in name else "-")
    ax_sc.plot([live["n_hits"]], [live["seconds"]], "*", ms=13, color="#2563eb", mec=INK, mew=0.6, label="this run")
    ax_sc.plot(
        [live["n_hits"]], [live["seconds_h"]], "*", ms=12, color=SOLVERS["Hungarian (ours; PyTorch)"], mec=INK, mew=0.6
    )
    ts = [rw["wall_s"] for rw in bench if isinstance(rw["wall_s"], float)] + [live["seconds"], live["seconds_h"]]
    ax_sc.set_xlim(min(sizes) * 0.7, max(sizes) * 1.4)
    ax_sc.set_ylim(min(ts) * 0.4, max(ts) * 3e3)
    ax_sc.set_xlabel("hits in the wedge $N$ (dimensionless)")
    ax_sc.set_ylabel("wall time (s)")
    ax_sc.legend(
        loc="upper left",
        ncol=2,
        handlelength=0.8,
        columnspacing=0.5,
        handletextpad=0.3,
        borderpad=0.3,
        labelspacing=0.25,
    )
    ax_sc.set_title("wall time vs size (offline)")

    # gap / efficiency / fake per solver, at this wedge (or the nearest benchmark wedge)
    grid_dphi = sorted({rw["dphi_deg"] for rw in bench})
    near = min(grid_dphi, key=lambda d: abs(d - dphi))
    rows = {rw["solver"]: rw for rw in bench if rw["dphi_deg"] == near and rw["solver"] != "HiGHS LP"}
    names = [s for s in SOLVERS if s in rows]
    gaps = [max(100 * rows[s]["gap"], 0.0) for s in names]
    effs = [100 * rows[s]["efficiency"] for s in names]
    fakes = [100 * rows[s]["fake_rate"] for s in names]
    yb = np.arange(len(names))
    ax_gap.barh(yb, gaps, color=[SOLVERS[s] for s in names])
    ax_gap.set_yticks(yb, [SHORT[s] for s in names])
    ax_gap.invert_yaxis()
    ax_gap.set_xlim(0, max(max(gaps), 0.1) * 2.6)
    ax_gap.set_xlabel("gap to optimum (%)")
    for yy, g in zip(yb, gaps):
        ax_gap.text(g, yy, " 0 (exact)" if g < 1e-6 else f" {g:.2f}%", va="center", fontsize=9)
    w = 0.4
    ax_eff.barh(yb - w / 2, effs, height=w, color=[SOLVERS[s] for s in names])
    ax_eff.barh(yb + w / 2, fakes, height=w, color=[SOLVERS[s] for s in names], alpha=0.45, hatch="///")
    ax_eff.set_yticks(yb, [""] * len(names))
    ax_eff.invert_yaxis()
    ax_eff.set_xlim(0, 100)
    ax_eff.set_xlabel("efficiency / fake (%)")
    ax_eff.text(
        0.98,
        1.02,
        "solid: efficiency\nhatched: fake rate",
        transform=ax_eff.transAxes,
        ha="right",
        va="bottom",
        fontsize=9,
    )
    ax_gap.set_title(f"solvers at Δφ = {near:g}°", loc="left")

    # TrackML score vs time per event
    sota = tensors.load_table(tensors.SOTA)
    ax_ml.set_xscale("log")
    ax_ml.set_xlim(1e-3, 1e3)
    ax_ml.set_ylim(0.3, 1.05)
    ax_ml.set_xlabel("time per event (s)")
    ax_ml.set_ylabel("TrackML score (dimensionless)")
    labelled = {"Mikado": (-30, -16), "FASTrack": (2, 6), "Exa.TrkX": (6, -5)}
    for rw in sota:
        name = rw["method"].split(" (")[0]
        if isinstance(rw["time_s"], float):
            ax_ml.plot(rw["time_s"], rw["score"], "D", ms=6, color=SOTA_COLOUR)
            if name in labelled:
                ax_ml.annotate(
                    name,
                    (rw["time_s"], rw["score"]),
                    fontsize=9,
                    xytext=labelled[name],
                    textcoords="offset points",
                    color="#14532d",
                )
        else:
            ax_ml.axhline(rw["score"], color=SOTA_COLOUR, lw=0.8, ls=":")
            if name == "Top Quarks":
                ax_ml.text(1.3e-3, rw["score"] + 0.008, "Top Quarks (not timed)", fontsize=9, color="#14532d")
    full = [rw for rw in bench if rw["dphi_deg"] == max(grid_dphi) and isinstance(rw["trackml_score"], float)]
    for rw in full:
        ax_ml.plot(rw["wall_s"], rw["trackml_score"], "o", ms=6, color=SOLVERS[rw["solver"]], mec=INK, mew=0.4)
    ax_ml.plot([live["seconds"]], [live["score"]], "*", ms=13, color="#2563eb", mec=INK, mew=0.6)
    ax_ml.plot(
        [live["seconds_h"]], [live["score_h"]], "*", ms=12, color=SOLVERS["Hungarian (ours; PyTorch)"], mec=INK, mew=0.6
    )
    keys = [
        plt.Line2D([], [], ls="", marker="D", ms=6, color=SOTA_COLOUR, label="cited SOTA (full events)"),
        plt.Line2D([], [], ls="", marker="o", ms=6, color=GREY, mec=INK, label="ours, 360° subset"),
        plt.Line2D([], [], ls="", marker="*", ms=11, color=GREY, mec=INK, label="this run"),
    ]
    ax_ml.legend(handles=keys, loc="center left", bbox_to_anchor=(0.0, 0.66), frameon=False)
    ax_ml.set_title("TrackML score vs time")

    # frames: 0..N_FADE-1 fade in, then PDLP states, then hold
    total = N_FADE + len(states) + N_HOLD
    stamp = fig.text(0.5, 0.012, "", ha="center", fontsize=10, color="#374151")

    def draw_residuals(iteration):
        upto = it <= iteration
        upto[: min(2, len(upto))] = True  # the first checks are visible from frame 0
        for line, key in ((l_p, "primal"), (l_d, "dual"), (l_g, "gap")):
            line.set_data(it[upto], np.maximum(hist[key][upto], floor))

    def update(f):
        if f < N_FADE:
            alpha = 0.2 * (f + 1) / N_FADE
            cand_xy.set_alpha(alpha)
            cand_rz.set_alpha(alpha)
            title_xy.set_text(f"hits and candidates, Δφ = {dphi:g}°")
            draw_residuals(it[0])
            res_title.set_text(f"PDLP residuals, iteration {int(it[min(1, len(it) - 1)])}")
            return
        k = min(f - N_FADE, len(states) - 1)
        iteration, xy, rz, colours, n_trip = states[k]
        cand_xy.set_alpha(0.06)
        cand_rz.set_alpha(0.06)
        trk_xy.set_segments(xy)
        trk_xy.set_color(colours)
        trk_rz.set_segments(rz)
        trk_rz.set_color(colours)
        draw_residuals(iteration)
        title_xy.set_text(f"tracks at iteration {iteration} ({n_trip} triplets)")
        res_title.set_text(f"PDLP residuals, iteration {iteration}")
        if f >= N_FADE + len(states):
            gap = (
                f"gap to HiGHS {100 * max(live['opt'] - live['obj'], 0) / live['opt']:.2f}%"
                if live["opt"]
                else (f"≤ {100 * max(live['bound'] - live['obj'], 0) / live['bound']:.2f}% from optimum (dual bound)")
            )
            stamp.set_text(
                f"PDLP + repair: {gap}, efficiency {100 * live['eff']:.1f}%, fake {100 * live['fake']:.1f}%, "
                f"score {live['score']:.3f}   |   Hungarian: efficiency {100 * live['eff_h']:.1f}%, "
                f"fake {100 * live['fake_h']:.1f}%, score {live['score_h']:.3f}"
            )

    frames = list(range(total))
    return _save_shared_palette(fig, update, frames, path)


def animate(*args, **kwargs) -> Path:
    """Render the GIF with readable font sizes (see RC)."""
    with plt.rc_context(RC):
        return _animate(*args, **kwargs)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'dphi': 30,
    'pt_min': 0.5,
    'max_iter': 5000,
    'digits': 6,
    'check': 32
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.ilp_track_finding")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.ilp_track_finding.main")
problem_dir = Path("problems/nuclear/ilp_track_finding")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
